
1. You provide a **latitude and longitude** (manually — Colab has no GPS)
2. The system queries **OpenStreetMap's Overpass API** for `tourism=attraction` within 5 km
3. It calculates the exact distance from your location to each place using **geopy**
4. Results are **ranked by distance** (closest first) and given a proximity score
5. An interactive **Folium map** is generated with all results plotted

In [ ]:
!pip install geopy folium groq gradio --quiet

##Import Libraries

In [ ]:
import requests                           # HTTP requests to Overpass API
import json                               # JSON parsing
import folium                             # Interactive map visualization
from geopy.distance import geodesic       # Accurate geographic distance
from IPython.display import display, HTML # Render map inline in Colab

##Set Your Location

In [ ]:
USER_LAT = 30.0444
USER_LON = 31.2357
SEARCH_RADIUS_METERS = 8000

In [ ]:
import time

# ── Three public Overpass mirrors, tried in order ───────────────────────────
OVERPASS_MIRRORS = [
    "https://overpass-api.de/api/interpreter",       # primary (Germany)
    "https://overpass.kumi.systems/api/interpreter",  # mirror 1 (Austria)
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",  # mirror 2 (Russia)
]

def get_places(lat: float, lon: float, radius: int = 8000) -> list:
    """
    Fetch tourist attractions from OpenStreetMap via the Overpass API.

    Reliability improvements over v1:
      • Tries 3 public mirrors in sequence
      • Retries each mirror up to 3 times with exponential backoff
        (2 s → 4 s → 8 s) before moving to the next mirror
      • Timeout raised to 60 s per attempt
      • Query timeout raised to 60 s (server-side)
      • Broader query: also fetches historic=* and museum=* nodes so the
        allowlist filter has more raw material to work with

    Returns a list of raw OSM elements, or [] if all mirrors fail.
    """
    # ------------------------------------------------------------------
    # Broader Overpass query — catches temples, tombs, monuments that are
    # tagged historic=* rather than tourism=attraction
    # ------------------------------------------------------------------
    overpass_query = f"""
[out:json][timeout:60];
(
  node["tourism"="attraction"](around:{radius},{lat},{lon});
  way["tourism"="attraction"](around:{radius},{lat},{lon});
  relation["tourism"="attraction"](around:{radius},{lat},{lon});
  node["historic"](around:{radius},{lat},{lon});
  way["historic"](around:{radius},{lat},{lon});
  relation["historic"](around:{radius},{lat},{lon});
  node["tourism"="museum"](around:{radius},{lat},{lon});
  way["tourism"="museum"](around:{radius},{lat},{lon});
  node["leisure"~"park|garden|nature_reserve"](around:{radius},{lat},{lon});
  way["leisure"~"park|garden|nature_reserve"](around:{radius},{lat},{lon});
  node["amenity"="place_of_worship"](around:{radius},{lat},{lon});
  way["amenity"="place_of_worship"](around:{radius},{lat},{lon});
  node["man_made"~"lighthouse|obelisk|tower|landmark"](around:{radius},{lat},{lon});
  way["man_made"~"lighthouse|obelisk|tower|landmark"](around:{radius},{lat},{lon});
);
out center;
"""

    MAX_RETRIES   = 3       # attempts per mirror
    BASE_WAIT     = 2       # seconds (doubles each retry: 2, 4, 8)
    REQUEST_TIMEOUT = 60    # seconds per HTTP request

    for mirror_url in OVERPASS_MIRRORS:
        mirror_name = mirror_url.split("/")[2]   # just the hostname

        for attempt in range(1, MAX_RETRIES + 1):
            wait = BASE_WAIT ** attempt           # 2, 4, 8 seconds

            try:
                print(f"[{mirror_name}] Attempt {attempt}/{MAX_RETRIES} ...", end=" ")

                response = requests.post(
                    mirror_url,
                    data={"data": overpass_query},
                    timeout=REQUEST_TIMEOUT,
                    headers={"User-Agent": "AtharAI/1.0 (graduation project)"}
                )
                response.raise_for_status()

                data     = response.json()
                elements = data.get("elements", [])

                print(f"{len(elements)} raw elements received.")
                return elements

            except requests.exceptions.Timeout:
                print(f"Timed out.")
                if attempt < MAX_RETRIES:
                    print(f"    Waiting {wait}s before retry...")
                    time.sleep(wait)

            except requests.exceptions.HTTPError as e:
                # 429 = too many requests; 504 = gateway timeout from mirror
                code = e.response.status_code if e.response else "?"
                print(f"HTTP {code}.")
                if attempt < MAX_RETRIES:
                    print(f"    Waiting {wait}s before retry...")
                    time.sleep(wait)

            except requests.exceptions.RequestException as e:
                print(f"Network error: {e}")
                break   # Non-recoverable for this mirror; try next one

            except Exception as e:
                print(f"Unexpected error: {e}")
                break

        print(f"    ↪ Switching to next mirror...")

    # All mirrors failed
    print()
    print("All Overpass mirrors failed.")
    print("   Possible causes:")
    print("   • Overpass API is under heavy load — wait 2–3 minutes and re-run")
    print("   • Colab network is restricted (try Runtime > Restart and run all)")
    print("   • Your IP is temporarily rate-limited — try changing the query radius")
    return []


## Distance Calculation

In [ ]:
def calculate_distance(user_lat: float, user_lon: float,
                        place_lat: float, place_lon: float) -> float:
    """
    Geodesic distance between user and a place (geopy, ellipsoidal Earth).
    Returns km rounded to 2 dp.
    """
    return round(geodesic((user_lat, user_lon), (place_lat, place_lon)).kilometers, 2)


##Data Cleaning & Deduplication Helpers

Seven helper functions used internally by `filter_and_rank_places()`.

### Noise rejection — 5 gates (all must pass to keep a place)

| Gate | Check | Example catches |
|---|---|---|
| 1 | OSM tag allowlist | Anything without `historic=*`, `tourism=attraction/museum` etc. |
| 2 | Pattern blocklist | `Home`, `Museum Storage`, `Excavation Area`, `Pavillion` |
| 3 | Arabic short name | `قيس عبه` (2 words × ≤4 chars each) |
| 4 | Bare English name | `Essam`, `Mohamed Awad` (≤2 words, no touristic qualifier) |
| 5 | Keyword match | EN: `cinema`, `shisha`, `mogamma` · AR: `قهوة`, `شارع`, `نادى` |

In [ ]:
import re
import unicodedata

NOISE_KEYWORDS_EN = {
    # Finance
    "bank", "atm", "exchange", "forex",
    # Food & drink
    "restaurant", "cafe", "coffee", "bakery", "shisha", "hookah", "narghile",
    # Accommodation
    "hotel", "hostel", "motel", "resort", "lodge", "guesthouse",
    # Retail
    "shop", "store", "market", "supermarket", "mall", "boutique",
    "souvenir", "bazaar", "alabaster", "tailleurs", "jewel",
    # Health
    "pharmacy", "hospital", "clinic", "dental", "medical",
    # Transport / services
    "bus", "ferry", "terminal", "station", "stop", "airport",
    "taxi", "balloon", "boating", "boat", "boats", "fishing",
    # Government / admin
    "office", "government", "ministry", "embassy", "consulate",
    "municipality", "governorate", "diwan", "mogamma",
    # Education
    "school", "university", "college", "institute",
    # Events / venues
    "cinema", "theatre", "theater", "club", "conference", "congress",
    # Infrastructure / misc
    "parking", "toilet", "restroom", "fuel", "gas", "petrol",
    "police", "fire", "post", "storage",
    # Navigation noise
    "street", "avenue", "road",
}


NOISE_KEYWORDS_AR = [
    "محافظة", "ديوان", "وزارة", "سفارة", "حكومة", "مجلس", "إدارة", "نيابة",
    "قاعة المؤتمرات", "قاعة اجتماعات", "مؤتمر",
    "مدرسة", "جامعة", "كلية", "معهد",
    "مستشفى", "عيادة", "صيدلية",
    "مطعم", "مقهى", "قهوة", "شيشة",
    "محل", "متجر", "سوبرماركت",
    "بنك", "صراف",
    "موقف", "محطة", "مطار",
    "فندق", "نزل",
    "نادى", "نادي", "ملعب",
    "بيت ",     # trailing space: private home, not Bayt Al-Suhaymi style
    "شارع",
    "ميدان",
    "حى ",
    "حي ",
    "أولاد", "شركة", "مكتب",
]

NAME_BLOCKLIST_PATTERNS = [
    r"^home$",
    r"^site of the birth",
    r"^birth of",
    r"storage$",
    r"^excavation",
    r"^side gate$",
    r"^palace gate$",
    r"pavill?ion",
    r"^fisher boats?$",
    r"^dok dok",
    r"\bboating\b",
]


TOURISTIC_QUALIFIERS = {
    "museum", "palace", "mosque", "church", "tomb", "temple", "monument",
    "mausoleum", "citadel", "tower", "gate", "bridge", "garden", "park",
    "house", "villa", "library", "gallery", "memorial", "shrine",
    "monastery", "cathedral", "synagogue", "ruins", "castle", "fort",
    "fortress", "complex", "site", "heritage", "nilometer", "obelisk",
    "pyramid", "sphinx", "zoo", "aquarium", "panorama", "statue",
    "fountain", "island", "lake", "valley", "mountain", "cave", "village",
    "quarter", "corniche", "masjid", "bab", "burg", "wikala", "sabil",
    "kuttab", "madrasa", "takiyya", "wekala", "bayt", "dar", "qasr",
    "khan", "hammam", "zawiya", "habu", "ramesseum", "karnak", "fustat",
    "saqqara", "historic", "coptic", "pharaonic", "islamic", "roman",
    "military", "national", "royal", "agricultural", "geological",
    "tree", "wall", "courtyard", "foundation",
}

# Valid 2-word names that contain no qualifier word
TWO_WORD_PLACE_EXCEPTIONS = {
    "ramses ii", "coptic cairo", "zamalek tree", "historic cairo",
    "fustat", "chan el chalili", "banana island", "cairo lake",
    "cairo tower", "nile fountain",
}

TOURIST_ALLOWLIST = [
    {"tourism": {
        "attraction", "museum", "gallery", "artwork", "viewpoint",
        "archaeological_site", "ruins", "theme_park", "zoo", "aquarium",
    }},
    {"historic": set()},
    {"leisure": {"park", "nature_reserve", "garden"}},
    {"amenity": {"place_of_worship"}},
    {"man_made": {"lighthouse", "obelisk", "tower", "landmark"}},
    {"natural": {"peak", "volcano", "cave_entrance", "beach"}},
]


def _matches_allowlist(tags: dict) -> bool:
    for rule in TOURIST_ALLOWLIST:
        for key, accepted_values in rule.items():
            val = tags.get(key, "").lower()
            if val:
                if not accepted_values or val in accepted_values:
                    return True
    return False


def _extract_coords(element: dict):
    if element.get("type") == "node":
        return element.get("lat"), element.get("lon")
    center = element.get("center", {})
    return center.get("lat"), center.get("lon")


def _resolve_name(tags: dict) -> str | None:
    raw = (
        tags.get("name:en")
        or tags.get("int_name")
        or tags.get("name")
        or tags.get("name:ar")
        or tags.get("official_name")
    )
    return raw.strip() if raw and raw.strip() else None


def _clean_name(name: str) -> str:
    name = unicodedata.normalize("NFC", name)
    name = name.replace("_", " ").replace("-", " ")
    name = re.sub(r"\s+", " ", name).strip()
    if re.search(r"[A-Za-z]", name):
        name = name.title()
    return name


def _is_noise(name: str, tags: dict) -> bool:
    """
    Five-gate noise filter. Returns True (reject) if ANY gate fires.

    Gate 1 — OSM tag allowlist    : no confirmed touristic tag
    Gate 2 — Pattern blocklist    : known noise patterns
    Gate 3 — Arabic short name    : 2-word Arabic, both words ≤ 4 chars
    Gate 4 — Bare English name    : ≤ 2 Latin words, no touristic qualifier
    Gate 5 — Keyword match        : EN whole-word or AR substring noise keyword
    """
    name_lower = name.lower().strip()
    arabic_chars = len(re.findall(r"[\u0600-\u06FF]", name))

    # Gate 1
    if not _matches_allowlist(tags):
        return True

    # Gate 2
    for pat in NAME_BLOCKLIST_PATTERNS:
        if re.search(pat, name_lower):
            return True

    if arabic_chars > 0:
        ar_words = name.split()
        if len(ar_words) == 2 and all(len(w) <= 4 for w in ar_words):
            return True

    # Gate 4: English bare name (person name or vague label)
    latin_words = re.findall(r"[a-zA-Z']+", name_lower)
    if latin_words and arabic_chars == 0:
        has_qualifier = any(w in TOURISTIC_QUALIFIERS for w in latin_words)
        is_exception  = name_lower in TWO_WORD_PLACE_EXCEPTIONS
        if not has_qualifier and not is_exception and len(latin_words) <= 2:
            return True

    # Gate 5a: English keywords
    for kw in NOISE_KEYWORDS_EN:
        if re.search(rf"\b{re.escape(kw)}\b", name_lower):
            return True

    # Gate 5b: Arabic substrings
    for ar_kw in NOISE_KEYWORDS_AR:
        if ar_kw in name:
            return True

    if len(name.strip()) <= 1 or re.fullmatch(r"[\d\s.,/\-()]+", name):
        return True

    return False


def _coord_key(lat: float, lon: float, precision: int = 3) -> tuple:
    return (round(lat, precision), round(lon, precision))


def _deduplicate(candidates: list) -> list:
    by_name: dict[str, dict] = {}
    for place in candidates:
        key = place["name"].lower()
        if key not in by_name or place["distance_km"] < by_name[key]["distance_km"]:
            by_name[key] = place

    seen_coords: dict[tuple, int] = {}
    kept: list[dict] = []
    for place in by_name.values():
        ck = _coord_key(place["lat"], place["lon"])
        if ck in seen_coords:
            idx = seen_coords[ck]
            if place["distance_km"] < kept[idx]["distance_km"]:
                best_name = max(kept[idx]["name"], place["name"], key=len)
                kept[idx] = {**place, "name": best_name}
        else:
            seen_coords[ck] = len(kept)
            kept.append(place)
    return kept


print("Cleaning helpers loaded (v4 - 5-gate filter):")
print("  Gate 1: OSM tag allowlist")
print("  Gate 2: Pattern blocklist  (home, storage, excavation, ...)")
print("  Gate 3: Arabic short name  (2 words, both <= 4 chars)")
print("  Gate 4: Bare English name  (<= 2 words, no touristic qualifier)")
print("  Gate 5: Keyword lists      (EN whole-word + AR substrings)")


Cleaning helpers loaded (v4 - 5-gate filter):
  Gate 1: OSM tag allowlist
  Gate 2: Pattern blocklist  (home, storage, excavation, ...)
  Gate 3: Arabic short name  (2 words, both <= 4 chars)
  Gate 4: Bare English name  (<= 2 words, no touristic qualifier)
  Gate 5: Keyword lists      (EN whole-word + AR substrings)


##Filter, Clean, Deduplicate & Rank

`filter_and_rank_places()` now runs a 6-step production-grade pipeline:

```
Raw OSM elements
      │
      ├─ 1. Extract coordinates       (node / way / relation)
      ├─ 2. Resolve & clean name      (priority order + Title Case)
      ├─ 3. Validate tourist relevance (noise keywords + OSM tag check)
      ├─ 4. Distance filter           (geodesic, skip > radius)
      ├─ 5. Deduplicate               (name match + ≤200 m spatial merge)
      └─ 6. Score & sort              (proximity 0–100, ascending dist)
```

In [ ]:
def filter_and_rank_places(elements: list, user_lat: float, user_lon: float,
                            max_radius_km: float = 8.0) -> list:
    """
    Production-grade pipeline: parse → clean → validate → deduplicate → rank.

    Steps
    ─────
    1. Extract coordinates  (node → direct lat/lon | way/relation → center)
    2. Resolve & clean name (priority: name:en > name > name:ar > official_name)
    3. Validate touristic relevance (noise keywords + OSM tag check)
    4. Compute geodesic distance; skip if outside radius
    5. Deduplicate by name (case-insensitive) then by coordinates (≤200 m)
    6. Compute proximity score and sort ascending by distance

    Returns
    ───────
    List of dicts: {name, distance_km, lat, lon, score}
    Sorted by distance_km ascending.
    """
    raw_before = len(elements)
    candidates = []

    for element in elements:
        tags = element.get("tags", {})

        # ── 1. Coordinates ───────────────────────────────────────────────
        place_lat, place_lon = _extract_coords(element)
        if place_lat is None or place_lon is None:
            continue

        # ── 2. Name resolution & cleaning ───────────────────────────────
        raw_name = _resolve_name(tags)
        if not raw_name:
            continue                    # No name at all → discard

        name = _clean_name(raw_name)

        # ── 3. Noise / relevance filter ──────────────────────────────────
        if _is_noise(name, tags):
            continue

        # ── 4. Distance filter ───────────────────────────────────────────
        dist_km = calculate_distance(user_lat, user_lon, place_lat, place_lon)
        if dist_km > max_radius_km:
            continue

        candidates.append({
            "name"        : name,
            "distance_km" : dist_km,
            "lat"         : place_lat,
            "lon"         : place_lon,
            "tag_count"   : len(tags),
        })

    # ── 5. Deduplicate (name + spatial) ─────────────────────────────────
    deduped = _deduplicate(candidates)


    #

    # ── 6. Score & sort ──────────────────────────────────────────────────
    results = []
    for place in deduped:
        prox  = (1 - place["distance_km"] / max_radius_km) * 70
        rich  = min(place.get("tag_count", 0) / 10, 1.0) * 30
        score = round(max(0.0, prox + rich), 1)
        results.append({**place, "score": score})

    # ── Diagnostic summary ───────────────────────────────────────────────
    print(f"Raw elements from API   : {raw_before}")
    print(f"After clean + validate  : {len(candidates)}")
    print(f"After deduplication     : {len(results)}")
    print(f"Final recommendations   : {len(results)}")

    return results


In [ ]:
from groq import Groq

groq_client = Groq(api_key="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")  # paste your real key here

def enrich_with_descriptions(places: list, top_n: int = 10) -> list:
    print(f"Enriching top {min(top_n, len(places))} places...")
    for i, place in enumerate(places[:top_n]):
        try:
            prompt = (
                f"In one sentence (max 20 words), describe the tourist attraction "
                f"'{place['name']}' in Egypt for a visitor. Be factual and specific."
            )
            resp = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=60,
            )
            raw = resp.choices[0].message.content
            print(f"  RAW TYPE: {type(raw)} | VALUE: {repr(raw)[:100]}")

            if isinstance(raw, list):
                desc = " ".join(
                    block.text if hasattr(block, "text") else str(block)
                    for block in raw
                ).strip()
            elif isinstance(raw, str):
                desc = raw.strip()
            else:
                desc = str(raw).strip()

            place["description"] = desc
            print(f"  #{i+1} {place['name'][:30]} → {desc[:60]}")

        except Exception as e:
            place["description"] = ""
            print(f"  #{i+1} FAILED: {e}")

    return places


# ── TEMPORARY TEST ──────────────────────────────────────────────
test_places = [
    {"name": "Gayer Anderson Museum", "distance_km": 2.29, "score": 80.0},
    {"name": "Nilometer In Rhoda Island", "distance_km": 4.21, "score": 51.2},
]
result = enrich_with_descriptions(test_places, top_n=2)
for p in result:
    print(f"\nNAME: {p['name']}")
    print(f"DESC: {p.get('description', 'MISSING')}")

Enriching top 2 places...
  RAW TYPE: <class 'str'> | VALUE: 'Gayer-Anderson Museum: 16th-century Egyptian house in Cairo.'
  #1 Gayer Anderson Museum → Gayer-Anderson Museum: 16th-century Egyptian house in Cairo.
  RAW TYPE: <class 'str'> | VALUE: "Ancient Egyptian Nilometer measures Nile River's water level."
  #2 Nilometer In Rhoda Island → Ancient Egyptian Nilometer measures Nile River's water level

NAME: Gayer Anderson Museum
DESC: Gayer-Anderson Museum: 16th-century Egyptian house in Cairo.

NAME: Nilometer In Rhoda Island
DESC: Ancient Egyptian Nilometer measures Nile River's water level.


##Map Visualization

In [ ]:
def build_map(user_lat: float, user_lon: float, places: list,
              radius_km: float = 8.0) -> folium.Map:
    """
    Build an interactive Folium map centered on the user's location.

    Adds:
      - Red marker  : user's position
      - Blue circle : search radius boundary
      - Colored pins: each attraction (green < 1km, blue < 3km, orange otherwise)
      - Popups      : name, distance, score, coordinates
    """
    m = folium.Map(
        location=[user_lat, user_lon],
        zoom_start=14,
        tiles="https://{s}.basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}{r}.png",
        attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> &copy; <a href="https://carto.com/">CARTO</a>'
    )

    # Draw search radius circle
    folium.Circle(
        location=[user_lat, user_lon],
        radius=radius_km * 1000,
        color="#3388FF",
        fill=True,
        fill_color="#3388FF",
        fill_opacity=0.05,
        dash_array="10",
        tooltip=f"Search radius: {radius_km} km"
    ).add_to(m)

    # User location marker
    folium.Marker(
        location=[user_lat, user_lon],
        popup=folium.Popup("<b>You are here</b>", max_width=150),
        tooltip="Your Location",
        icon=folium.Icon(color="red", icon="user", prefix="fa")
    ).add_to(m)

    # Attraction markers
    for rank, place in enumerate(places, start=1):
        popup_html = f"""
        <div style='font-family:Arial; min-width:180px;'>
            <b style='color:#1a73e8;'>#{rank} {place['name']}</b><br>
            <hr style='margin:4px 0'>
            Distance: <b>{place['distance_km']} km</b><br>
            Score: <b>{place['score']} / 100</b><br>
            ({place['lat']:.5f}, {place['lon']:.5f})
        </div>"""

        # Color by proximity
        if place["distance_km"] < 1.0:
            pin_color = "green"
        elif place["distance_km"] < 3.0:
            pin_color = "blue"
        else:
            pin_color = "orange"

        folium.Marker(
            location=[place["lat"], place["lon"]],
            popup=folium.Popup(popup_html, max_width=220),
            tooltip=f"#{rank} {place['name']} ({place['distance_km']} km)",
            icon=folium.Icon(color=pin_color, icon="camera", prefix="fa")
        ).add_to(m)

    return m

##Print Results Table

In [ ]:
def print_results(places: list, user_lat: float, user_lon: float) -> None:
    """Print a formatted ranked table of recommended attractions."""
    print(f"Recommended attractions near ({user_lat}, {user_lon}):")

    if not places:
        print("\nNo tourist attractions found within the search radius.")
        print("Try increasing the radius or choosing a different location.")
        return

    print(f"  {'#':<4} {'Name':<35} {'Distance':>10}  {'Score':>7}  Coordinates")

    for rank, place in enumerate(places, start=1):
        name   = place["name"][:34]
        dist   = f"{place['distance_km']} km"
        score  = f"{place['score']}/100"
        coords = f"({place['lat']:.4f}, {place['lon']:.4f})"
        desc   = place.get("description", "")
        print(f"  {rank:<4} {name:<35} {dist:>10}  {score:>7}  {coords}")
        if desc:
            print(f"        📍 {desc}")



##Run the Full Pipeline

In [ ]:
import pandas as pd

#print top 20
pd.set_option('display.max_rows', 20)


# STEP A: Fetch from OpenStreetMap
raw_elements = get_places(
    lat=USER_LAT, lon=USER_LON, radius=SEARCH_RADIUS_METERS
)

# STEP B: Filter, clean, deduplicate, rank
recommended_places = filter_and_rank_places(
    elements=raw_elements,
    user_lat=USER_LAT,
    user_lon=USER_LON,
    max_radius_km=SEARCH_RADIUS_METERS / 1000
)


# STEP C: Enrich with LLM descriptions
recommended_places = enrich_with_descriptions(recommended_places, top_n=10)

# STEP D: Print results first 30 only
print_results(places=recommended_places[:10], user_lat=USER_LAT, user_lon=USER_LON)

# print_results(places=recommended_places, user_lat=USER_LAT, user_lon=USER_LON)

[overpass-api.de] Attempt 1/3 ... 7746 raw elements received.
Raw elements from API   : 7746
After clean + validate  : 430
After deduplication     : 349
Final recommendations   : 349
Enriching top 10 places...
  RAW TYPE: <class 'str'> | VALUE: "Burg Al Zafr: ancient fortress in Egypt's Sinai Peninsula."
  #1 Burg Al Zafr → Burg Al Zafr: ancient fortress in Egypt's Sinai Peninsula.
  RAW TYPE: <class 'str'> | VALUE: 'Historic mosque in Cairo, Egypt, built in 1360 AD.'
  #2 Aslam Al Selehdar Mosque → Historic mosque in Cairo, Egypt, built in 1360 AD.
  RAW TYPE: <class 'str'> | VALUE: 'Ancient Christian church in Coptic Cairo, Egypt, dating back to 6th century AD.'
  #3 Saint Mercurius Church In Copt → Ancient Christian church in Coptic Cairo, Egypt, dating back
  RAW TYPE: <class 'str'> | VALUE: 'Gayer-Anderson Museum: 17th-century Ottoman house in Cairo.'
  #4 Gayer Anderson Museum → Gayer-Anderson Museum: 17th-century Ottoman house in Cairo.
  RAW TYPE: <class 'str'> | VALUE: "Histor

##Display Interactive Map

In [ ]:
#display only top 10
tourist_map = build_map(
    user_lat=USER_LAT,
    user_lon=USER_LON,
    places=recommended_places[:10],
    radius_km=SEARCH_RADIUS_METERS / 1000
)

tourist_map

In [ ]:
# ── Gradio UI ────────────────────────────────────────────────────────────────
import gradio as gr

_last_map_path = "/tmp/athar_map.html"

def run_pipeline(lat: float, lon: float, radius_km: float):
    global _last_map_path

    # Step 1: Fetch
    elements = get_places(lat, lon, radius=int(radius_km * 1000))
    if not elements:
        return "No data returned from API. Try again in a moment.", gr.update(visible=False)

    # Step 2: Filter & rank
    places = filter_and_rank_places(elements, lat, lon, max_radius_km=radius_km)
    if not places:
        return "No attractions found. Try a larger radius.", gr.update(visible=False)

    # Step 3: Enrich top 10 with LLM descriptions (same as pipeline cell)
    places = enrich_with_descriptions(places, top_n=10)

    # Step 4: Build results text — top 10 only, same format as print_results
    lines = []
    for rank, p in enumerate(places[:10], 1):
        name   = p["name"][:34]
        dist   = f"{p['distance_km']} km"
        score  = f"{p['score']}/100"
        coords = f"({p['lat']:.4f}, {p['lon']:.4f})"
        desc   = p.get("description", "").strip()

        line = f"#{rank:<3} {name:<35} | {dist:>8} | Score: {score:>8} | {coords}"
        if desc:
            line += f"\n      📍 {desc}"
        lines.append(line)

    results_text = "\n\n".join(lines)

    # Step 5: Build map with ONLY the top 10 places (same as results)
    m = build_map(lat, lon, places[:10], radius_km)
    m.save(_last_map_path)

    return results_text, gr.update(visible=True)


def open_map():
    try:
        with open(_last_map_path, "r", encoding="utf-8") as f:
            raw = f.read()
        srcdoc = raw.replace("&", "&amp;").replace('"', "&quot;")
        iframe = (
            f'<iframe srcdoc="{srcdoc}" '
            f'width="100%" height="600px" '
            f'frameborder="0" allowfullscreen></iframe>'
        )
        return gr.update(value=iframe, visible=True)
    except FileNotFoundError:
        return gr.update(value="<p>No map found. Run the search first.</p>", visible=True)


with gr.Blocks(title="AtharAI", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🗺️ AtharAI — Tourist Attraction Finder
    Enter your coordinates and search radius, then click **Find Attractions**.
    """)

    with gr.Row():
        lat_in = gr.Number(value=30.0444, label="Latitude",  precision=6)
        lon_in = gr.Number(value=31.2357, label="Longitude", precision=6)
        rad_in = gr.Slider(minimum=1, maximum=15, value=8, step=1, label="Radius (km)")

    find_btn = gr.Button("🔍 Find Attractions", variant="primary", size="lg")

    out_txt = gr.Textbox(
        label="Top 10 Attractions",
        lines=30,
        placeholder="Results will appear here...",
    )

    map_btn = gr.Button("🗺️ Display Map", variant="secondary", size="lg", visible=False)

    out_map = gr.HTML(value="", visible=False)

    find_btn.click(
        fn=run_pipeline,
        inputs=[lat_in, lon_in, rad_in],
        outputs=[out_txt, map_btn]
    )

    map_btn.click(
        fn=open_map,
        inputs=[],
        outputs=[out_map]
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_20802/4008816367.py:60: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="AtharAI", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b88384f465de18ec05.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[overpass-api.de] Attempt 1/3 ... 7746 raw elements received.
Raw elements from API   : 7746
After clean + validate  : 430
After deduplication     : 349
Final recommendations   : 349
Enriching top 10 places...
  RAW TYPE: <class 'str'> | VALUE: "Burg Al Zafr: ancient mudbrick fortress in Egypt's Sinai Peninsula."
  #1 Burg Al Zafr → Burg Al Zafr: ancient mudbrick fortress in Egypt's Sinai Pen
  RAW TYPE: <class 'str'> | VALUE: 'Historic mosque in Islamic Cairo, Egypt, built 1347.'
  #2 Aslam Al Selehdar Mosque → Historic mosque in Islamic Cairo, Egypt, built 1347.
  RAW TYPE: <class 'str'> | VALUE: 'Ancient Coptic church in Cairo, Egypt, dating back to 6th century AD.'
  #3 Saint Mercurius Church In Copt → Ancient Coptic church in Cairo, Egypt, dating back to 6th ce
  RAW TYPE: <class 'str'> | VALUE: 'Gayer-Anderson Museum is a historic house museum in Cairo, Egypt.'
  #4 Gayer Anderson Museum → Gayer-Anderson Museum is a historic house museum in Cairo, E
  RAW TYPE: <class 'str'> | VA

---

## 📘 Architecture Summary

```
User Input (lat, lon)
        │
        ▼
get_places(lat, lon)
  └─ Overpass API: nodes + ways + relations, tourism=attraction, 5 km radius
  └─ Returns raw JSON elements
        │
        ▼
filter_and_rank_places(elements, user_lat, user_lon)
  └─ Extract coords  (node → direct lat/lon | way/relation → center dict)
  └─ Resolve name    (name → name:en → name:ar → "Unknown Place")
  └─ calculate_distance() using geopy geodesic
  └─ Filter > 5 km
  └─ Proximity score = (1 - dist/radius) × 100
  └─ Sort ascending by distance
        │
        ├──► print_results()  →  Ranked table in console
        │
        └──► build_map()  →  Folium interactive map
               └─ Red pin: user location
               └─ Dashed circle: 5 km radius
               └─ Colored pins: green < 1km | blue < 3km | orange otherwise
               └─ Popups: name + distance + score
```

---
**AtharAI — Faculty of Artificial Intelligence, Graduation Project 2025**